<h1><strong>Problem Statement</strong></h1>


<h2>Business Context</h2>


<p>A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.</p>
<p>Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.</p>


<h2>Objective</h2>


<p>SuperKart is a retail chain operating supermarkets and food marts across various tier cities, offering a wide range of products. To optimize its inventory management and make informed decisions around regional sales strategies, SuperKart wants to accurately forecast the sales revenue of its outlets for the upcoming quarter.</p>
<p>To operationalize these insights at scale, the company has partnered with a data science firm—not just to build a predictive model based on historical sales data, but to develop and deploy a robust forecasting solution that can be integrated into SuperKart’s decision-making systems and used across its network of stores.</p>


<h2>Data Description</h2>


<p>The data contains the different attributes of the various products and stores.The detailed data dictionary is given below.</p>
<ul>
<li><strong>Product_Id</strong> - unique identifier of each product, each identifier having two letters at the beginning followed by a number.</li>
<li><strong>Product_Weight</strong> - weight of each product</li>
<li><strong>Product_Sugar_Content</strong> - sugar content of each product like low sugar, regular and no sugar</li>
<li><strong>Product_Allocated_Area</strong> - ratio of the allocated display area of each product to the total display area of all the products in a store</li>
<li><strong>Product_Type</strong> - broad category for each product like meat, snack foods, hard drinks, dairy, canned, soft drinks, health and hygiene, baking goods, bread, breakfast, frozen foods, fruits and vegetables, household, seafood, starchy foods, others</li>
<li><strong>Product_MRP</strong> - maximum retail price of each product</li>
<li><strong>Store_Id</strong> - unique identifier of each store</li>
<li><strong>Store_Establishment_Year</strong> - year in which the store was established</li>
<li><strong>Store_Size</strong> - size of the store depending on sq. feet like high, medium and low</li>
<li><strong>Store_Location_City_Type</strong> - type of city in which the store is located like Tier 1, Tier 2 and Tier 3. Tier 1 consists of cities where the standard of living is comparatively higher than its Tier 2 and Tier 3 counterparts.</li>
<li><strong>Store_Type</strong> - type of store depending on the products that are being sold there like Departmental Store, Supermarket Type 1, Supermarket Type 2 and Food Mart</li>
<li><strong>Product_Store_Sales_Total</strong> - total revenue generated by the sale of that particular product in that particular store</li>
</ul>


<h1><strong>Installing and Importing the necessary libraries</strong></h1>


In [ ]:
#Installing the libraries with the specified versions
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 requests==2.32.4 huggingface_hub==0.34.0 -q


<p><strong>Note:</strong></p>
<ul>
<li>
<p>After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.</p>
</li>
<li>
<p>On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.</p>
</li>
</ul>


In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# For splitting the dataset
from sklearn.model_selection import train_test_split

# Libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 100)

# Libraries different ensemble classifiers
from sklearn.ensemble import (
    BaggingRegressor,
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor,
)
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor

# Libraries to get different metric scores
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error
)

# To create the pipeline
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline,Pipeline

# To tune different models and standardize
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder

# To serialize the model
import joblib

# os related functionalities
import os

# API request
import requests

# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi


<h1><strong>Loading the dataset</strong></h1>


In [ ]:
# read the data
df = pd.read_csv("SuperKart.csv")


In [ ]:
data = df.copy()


<h1><strong>Data Overview</strong></h1>


In [ ]:
# returns the first 5 rows
data.head()


In [ ]:
print('The data has',data.shape[0],'rows and',data.shape[1],'columns')


In [ ]:
data.info()


In [ ]:
unique = data['Product_Sugar_Content'].value_counts()
print(unique)


In [ ]:
data.describe().T


In [ ]:
data.isnull().sum()


<p>There are no missing values</p>


<h1><strong>Exploratory Data Analysis (EDA)</strong></h1>


In [ ]:
def histogram_boxplot(data, feature, figsize=(15, 10), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (15,10))
    kde: whether to show the density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a triangle will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram


In [ ]:
# function to create labeled barplots

def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 2, 6))
    else:
        plt.figure(figsize=(n + 2, 6))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n],
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot


<h2>Univariate Analysis</h2>


In [ ]:
histogram_boxplot(df, "Product_Weight")


<p>median seems to be near 12.5 for product weight</p>


In [ ]:
labeled_barplot(data, "Product_Sugar_Content", perc=True)


<p>We can see that most of the product type has low sugar content from the above 2 chart</p>


In [ ]:
histogram_boxplot(df, "Product_Allocated_Area")


<p>distribution seems to be right skewed</p>


In [ ]:
histogram_boxplot(df, "Product_MRP")


<p>MRP seems to have a normal distribution with median at 150</p>


In [ ]:
histogram_boxplot(data, "Store_Establishment_Year")


<p>There seems to be more stores that are established recently</p>


In [ ]:
labeled_barplot(df, "Product_Type", perc=True)


<p>Highest number of products are 'Fruits and Vegetables' with 14.3% followed by 'Snack Foods' with 13.1% and son on.  The least type is 'Seafood' with only 0.9%</p>


In [ ]:
labeled_barplot(data, "Store_Id", perc=True)


In [ ]:
labeled_barplot(data, "Store_Type", perc=True)


<p>The Store_Id and Store_Type columns show similar distributions and top frequencies, suggesting a strong correlation or direct mapping between particular store IDs and their types.
The Store_Id 'OUT004' and Store_Type 'Supermarket Type2' both occur with the same frequency of 53.4%.</p>


In [ ]:
histogram_boxplot(df, "Product_Store_Sales_Total")


<p>The distribution for total sales is normally distributed with a slight tail to the left.</p>


In [ ]:
labeled_barplot(data, "Product_Type", perc=True)


<p>More products are sold in stores that are located in Tier 2 cities.</p>


In [ ]:
labeled_barplot(data, "Store_Size", perc=True)


<p>Medium-sized stores generated more sales</p>


<h2>Bivariate Analysis</h2>


In [ ]:
# creating a list of numerical columns
num_cols = df.select_dtypes(include=np.number).columns.tolist()

# dropping start and finish year from list of numerical columns as they are not numerical in nature
num_cols.remove("Store_Establishment_Year")

plt.figure(figsize=(12, 7))
sns.heatmap(
    df[num_cols].corr(), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="coolwarm"
)
plt.show()


In [ ]:
g = sns.pairplot(
    data,
    x_vars=["Product_Weight", "Product_Allocated_Area", "Product_MRP", "Store_Establishment_Year"],
    y_vars=["Product_Store_Sales_Total"],
    height=7,
    aspect=0.7,
)
g.fig.suptitle("Pairplot of Numerical Features and Target Variable", y=1.02)
plt.show()


<p>There is a positive correlation between Product_Store_Sales_Total and Product_Weigh and Product_MRP.</p>


In [ ]:
# Group by 'Store_Type' and calculate the average revenue for each store type
store_type_revenue = df.groupby('Store_Type')['Product_Store_Sales_Total'].sum()

# Group by 'Store_Location_City_Type' and calculate the average revenue for each location
location_revenue = df.groupby('Store_Location_City_Type')['Product_Store_Sales_Total'].sum()

# Find the store type and location with the highest average revenue
top_store_type = store_type_revenue.idxmax()
top_store_type_revenue = store_type_revenue.max()

top_location = location_revenue.idxmax()
top_location_revenue = location_revenue.max()

print(f"The type of store with the greatest revenue is {top_store_type}")
print(f"Overall sales for the best store type is {top_store_type_revenue:.2f}")

print(f"The location which generates the highest amount of revenue is {top_location}")
print(f"Revenue for the top location type is {top_location_revenue:.2f}")

# Creating a pivot
store_location_revenue = df.pivot_table(index='Store_Type', columns='Store_Location_City_Type', values='Product_Store_Sales_Total', aggfunc='sum')

# Create a bar plot
plt.figure(figsize=(12, 6))
store_location_revenue.plot(kind='bar', stacked=True, color=['#9fc5e8', '#a2c4c9', '#d5a6bd'])
plt.xlabel('Store Type')
plt.ylabel('Total Revenue')
plt.title('Impact of Store Type and Location on Revenue')
plt.xticks(rotation=0)
plt.legend(title='Location')
plt.tight_layout()
plt.show()


<p>As noticed above Supermarket Type 2 is the store type with the highest revenue. The top store type has an overall sales of USD 15427583.43. Tier 2 is the location where the most revenue is made. The revenue for top location type is USD 21650696.61</p>


In [ ]:
# Filter the DataFrame to rows with 'Product_Sugar_Content' set to 'Low'.
low_sugar_df = df[df['Product_Sugar_Content'] == 'Low Sugar']

# Count the quantity of each product type sold after grouping by 'Product_Type'
items_sold_by_product_type = low_sugar_df.groupby('Product_Type')['Product_Store_Sales_Total'].count().sort_values(ascending=False)

print(items_sold_by_product_type)

# Create a bar plot
plt.figure(figsize=(12, 6))
items_sold_by_product_type.plot(kind='bar', color=['#9fc5e8'])
plt.xlabel('Product Type')
plt.ylabel('Total Sales')
plt.title('Total Sales of Product Types with Low Sugar Content')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


<p>Number of items sold in each of the 16 product types that have low sugar content
We can see that The category "Fruits and Vegetables" saw sales of 864 items. 804 items were sold under the "Snack Foods" category etc</p>


In [ ]:
# Group by 'Store_Id' and 'Product_Type' to get the count of sales of each product type sold in each store
product_count_by_store = df.groupby(['Store_Id', 'Product_Type'])['Product_Store_Sales_Total'].count()

# Find the product type that has been sold the most number of times in each store
max_sold_product_by_store = product_count_by_store.groupby(level='Store_Id').idxmax()

print("Product type sold the most number of times in each store:")
print(max_sold_product_by_store)

# Group by 'Store_Id' and 'Product_Type' to get the total revenue of each product type in each store
revenue_by_product_type = df.groupby(['Store_Id', 'Product_Type'])['Product_Store_Sales_Total'].sum()

# Find the product type that is contributing the most to the revenue of each store
top_revenue_product_by_store = revenue_by_product_type.groupby(level='Store_Id').idxmax()

print("\nProduct type contributing the most to the revenue of each store:")
print(top_revenue_product_by_store)

sales_by_product_type_per_store = df.groupby(['Store_Id', 'Product_Type'])['Product_Store_Sales_Total'].sum().unstack()

# Plot the sales by product type per store using a grouped bar plot
fig, ax = plt.subplots(figsize=(20, 15))

# Plot the sales by product type per store using a grouped bar plot
sales_by_product_type_per_store.plot(kind='bar', ax=ax)
ax.set_xlabel('Store Id')
ax.set_ylabel('Total Sales')
ax.set_title('Sales by Product Type per Store')
ax.set_xticklabels(sales_by_product_type_per_store.index, rotation=45, ha='right')

# Adjust legend size
legend = ax.legend(title='Product Type', prop={'size': 20})

plt.tight_layout()
plt.show()


<p>The product type that is contributing the most to the revenue of the individual stores:
Snack Foods are the product type sold most frequently at the OUT001 store. Fruits and vegetables are the product category that OUT002 store sells the most of. Snack Foods are the product category that OUT003 store sells the most of. Fruits and vegetables are the product category that OUT004 store sells the most of. Similarly, Snack Foods is the product category that generates the most of the revenue for the OUT001 store. Fruits and vegetables are the product category for the OUT002 store that contributes the most to revenue. Snack Foods are the product category for the OUT003 store that generates the largest amount of revenue. Fruits and vegetables are the product category for the OUT004 store that contributes the most to revenue.</p>


<h2>Revenue Analysis by Categorical Features</h2>


In [ ]:
def plot_sales_by_category(dataframe, category_feature):
    """
    Groups the DataFrame by a specified categorical feature, calculates the sum of
    'Product_Store_Sales_Total', and plots the results as a bar chart.

    Args:
        dataframe (pd.DataFrame): The input DataFrame.
        category_feature (str): The name of the categorical column to group by.
    """
    df_revenue = dataframe.groupby([category_feature], as_index=False)[
        "Product_Store_Sales_Total"
    ].sum()
    # Sort the DataFrame by Product_Store_Sales_Total in descending order
    df_revenue = df_revenue.sort_values(by="Product_Store_Sales_Total", ascending=False)
    display(df_revenue)

    plt.figure(figsize=[8, 6])
    plt.xticks(rotation=90)
    sns.barplot(x=df_revenue[category_feature], y=df_revenue.Product_Store_Sales_Total, order=df_revenue[category_feature])
    plt.xlabel(category_feature.replace('_', ' '))
    plt.ylabel("Revenue")
    plt.title(f"Revenue by {category_feature.replace('_', ' ')}")
    plt.show()


In [ ]:
plot_sales_by_category(data, "Product_Sugar_Content")


<p>"Low Sugar" products generated more revenue.</p>


In [ ]:
plot_sales_by_category(data, "Store_Location_City_Type")


<p>Product Type: 'Fruits and Vegetables' generated the highest revenue among all product types.
Product Sugar Content: Products with 'Low Sugar' content contributed the most to the total revenue, aligning with their higher sales volume.
Store Performance: Store 'OUT004', identified as a 'Supermarket Type 2', recorded the highest revenue compared to other store IDs and types.
Store Size: Medium-sized stores were the highest revenue generators.
Store Location: Stores located in Tier 2 cities produced the most revenue.</p>


<h1><strong>Data Preprocessing</strong></h1>


<p>Product_Sugar_Content column conatins 4 types - "Low Sugar", "Regular", "No Sugar", "reg". Replace "reg" with "Regular" so fixing it</p>


In [ ]:
data["Product_Sugar_Content"].replace(to_replace=["reg"], value=["Regular"], inplace=True)


In [ ]:
data.Product_Sugar_Content.value_counts()


In [ ]:
plot_sales_by_category(data, "Product_Sugar_Content")


<p>We see that low sugar products has still high counts and we dont see any 'Reg'</p>


In [ ]:
data["Store_Age_Years"] = 2026 - data.Store_Establishment_Year


In [ ]:
data.head()


In [ ]:
data["Product_Type"].unique().tolist()


In [ ]:
perishables = [
    "Dairy",
    "Meat",
    "Fruits and Vegetables",
    "Breakfast",
    "Breads",
    "Seafood",
]


In [ ]:
def change(x):
    if x in perishables:
        return "Perishables"
    else:
        return "Non Perishables"


In [ ]:
data['Product_Type_Category'] = data['Product_Type'].apply(change)


In [ ]:
data.head()


In [ ]:
data.head()


In [ ]:
data["Product_Id"].info()


In [ ]:
# Extract the first two characters from the Product_Id column and store it in a new column
data["Product_Id_char"] = data["Product_Id"].str[:2]
data.head()


In [ ]:
data["Product_Id_char"].value_counts()


In [ ]:
display(data.loc[data.Product_Id_char == "FD", "Product_Type"].unique())
display(data.loc[data.Product_Id_char == "NC", "Product_Type"].unique())
display(data.loc[data.Product_Id_char == "DR", "Product_Type"].unique())


In [ ]:
data[["Product_Type_Category", "Product_Id_char"]].info()


In [ ]:
columns = ["Product_Type_Category", "Product_Id_char"]
for column in columns:
    data[column] = data[column].astype("category")


In [ ]:
data.columns.tolist()


In [ ]:
data = data.drop(["Product_Id", "Store_Id", "Product_Type", "Store_Establishment_Year"], axis=1)


In [ ]:
data.columns.tolist()


In [ ]:
for column in data.columns:
    if data[column].dtype == "object":
        data[column] = data[column].astype("category")


In [ ]:
data.describe(include='category').T


In [ ]:
data.head()


<h2>Data Preparation for Modelling</h2>


In [ ]:
X = data.drop("Product_Store_Sales_Total", axis=1)
y = data["Product_Store_Sales_Total"]

X.shape, y.shape


In [ ]:
# Split the data into train and test sets in 70:30 ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, shuffle=True
)


In [ ]:
X_train.shape, X_test.shape


In [ ]:
numerical_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["category"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)


In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = make_column_transformer(
    (StandardScaler(), numerical_features),
    (OneHotEncoder(handle_unknown="ignore"), categorical_features)
)

preprocessor


<h1><strong>Model Building</strong></h1>


<h2>Define functions for Model Evaluation</h2>


In [ ]:
# function to compute adjusted R-squared
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))

# function to compute different metrics to check performance of a regression model
def model_performance_regression(model, predictors, target):
    """
    Function to compute different metrics to check regression model performance

    model: regressor
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    r2 = r2_score(target, pred)  # to compute R-squared
    adjr2 = adj_r2_score(predictors, target, pred)  # to compute adjusted R-squared
    rmse = np.sqrt(mean_squared_error(target, pred))  # to compute RMSE
    mae = mean_absolute_error(target, pred)  # to compute MAE
    mape = mean_absolute_percentage_error(target, pred)  # to compute MAPE

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R-squared": r2,
            "Adj. R-squared": adjr2,
            "MAPE": mape,
        },
        index=[0],
    )

    return df_perf


<p>The ML models to be built can be any two out of the following:
1. Decision Tree
2. Bagging
3. Random Forest
4. AdaBoost
5. Gradient Boosting
6. XGBoost</p>


<p>For this use case, the best 2 to build and tune are usually:</p>
<p>RandomForestRegressor
XGBRegressor
Why these 2 fit best here</p>
<p>The data is tabular, mixed, and likely nonlinear.
Sales are influenced by interactions like price, store type, city tier, and display area.
Both models handle nonlinear relationships much better than simpler models.
They usually perform strongly on structured business datasets like this one.</p>


<h3>Train Model 1, Random Forest</h3>


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline

rf_estimator = RandomForestRegressor(random_state=1)
rf_estimator = make_pipeline(preprocessor, rf_estimator)
rf_estimator.fit(X_train, y_train)


<p>Evaluate Random Forest</p>


In [ ]:
rf_estimator_model_train_perf = model_performance_regression(rf_estimator, X_train, y_train)
rf_estimator_model_test_perf = model_performance_regression(rf_estimator, X_test, y_test)

print("Random Forest - Training Performance")
display(rf_estimator_model_train_perf)

print("Random Forest - Test Performance")
display(rf_estimator_model_test_perf)


<h3>Train Model 2, XGBoost</h3>


In [ ]:
from xgboost import XGBRegressor

xgb_estimator = XGBRegressor(random_state=1, objective="reg:squarederror")
xgb_estimator = make_pipeline(preprocessor, xgb_estimator)
xgb_estimator.fit(X_train, y_train)


<p>Evaluate XGBoost</p>


In [ ]:
xgb_estimator_model_train_perf = model_performance_regression(xgb_estimator, X_train, y_train)
xgb_estimator_model_test_perf = model_performance_regression(xgb_estimator, X_test, y_test)

print("XGBoost - Training Performance")
display(xgb_estimator_model_train_perf)

print("XGBoost - Test Performance")
display(xgb_estimator_model_test_perf)


<h1><strong>Model Performance Improvement - Hyperparameter Tuning</strong></h1>


<p>We will tune the two selected models, <code>RandomForestRegressor</code> and <code>XGBRegressor</code>, to improve generalization performance on the test set.</p>
<p>To keep the search practical, some hyperparameters are tested with 3 values and some with 2 values.</p>


In [ ]:
# Hyperparameter tuning for Random Forest

rf_tuned = RandomForestRegressor(random_state=1)
rf_tuned = make_pipeline(preprocessor, rf_tuned)

rf_params = {
    "randomforestregressor__n_estimators": [120, 180, 260],
    "randomforestregressor__max_depth": [8, 14, None],
    "randomforestregressor__min_samples_split": [2, 6],
    "randomforestregressor__min_samples_leaf": [1, 3],
    "randomforestregressor__max_features": ["sqrt", None],
}

rf_grid = GridSearchCV(
    rf_tuned,
    rf_params,
    scoring="r2",
    cv=5,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

rf_best = rf_grid.best_estimator_

print("Best Random Forest Parameters:")
print(rf_grid.best_params_)


<p>Evaluate tuned Random Forest</p>


In [ ]:
rf_tuned_model_train_perf = model_performance_regression(rf_best, X_train, y_train)
rf_tuned_model_test_perf = model_performance_regression(rf_best, X_test, y_test)

print("Tuned Random Forest - Training Performance")
display(rf_tuned_model_train_perf)

print("Tuned Random Forest - Test Performance")
display(rf_tuned_model_test_perf)


<p>Hyperparameter tuning for XGBoost</p>


In [ ]:
# Hyperparameter tuning for XGBoost

xgb_tuned = XGBRegressor(
    random_state=1,
    objective="reg:squarederror"
)
xgb_tuned = make_pipeline(preprocessor, xgb_tuned)

xgb_params = {
    "xgbregressor__n_estimators": [200, 320],
    "xgbregressor__max_depth": [3, 5, 7],
    "xgbregressor__learning_rate": [0.03, 0.08],
    "xgbregressor__subsample": [0.8, 1.0],
    "xgbregressor__colsample_bytree": [0.75, 1.0],
}

xgb_grid = GridSearchCV(
    xgb_tuned,
    xgb_params,
    scoring="r2",
    cv=3,
    n_jobs=-1
)

xgb_grid.fit(X_train, y_train)

xgb_best = xgb_grid.best_estimator_

print("Best XGBoost Parameters:")
print(xgb_grid.best_params_)


<p>Evaluate tuned XGBoost</p>


In [ ]:
xgb_tuned_model_train_perf = model_performance_regression(xgb_best, X_train, y_train)
xgb_tuned_model_test_perf = model_performance_regression(xgb_best, X_test, y_test)

print("Tuned XGBoost - Training Performance")
display(xgb_tuned_model_train_perf)

print("Tuned XGBoost - Test Performance")
display(xgb_tuned_model_test_perf)


<h1><strong>Model Performance Comparison, Final Model Selection, and Serialization</strong></h1>


In [ ]:
models_train_comp_df = pd.concat(
    [
        rf_estimator_model_train_perf.T,
        xgb_estimator_model_train_perf.T,
        rf_tuned_model_train_perf.T,
        xgb_tuned_model_train_perf.T,
    ],
    axis=1,
)

models_train_comp_df.columns = ["RF", "XGB", "Tuned RF", "Tuned XGB"]
print("Training performance comparison")
display(models_train_comp_df)

models_test_comp_df = pd.concat(
    [
        rf_estimator_model_test_perf.T,
        xgb_estimator_model_test_perf.T,
        rf_tuned_model_test_perf.T,
        xgb_tuned_model_test_perf.T,
    ],
    axis=1,
)

models_test_comp_df.columns = ["RF", "XGB", "Tuned RF", "Tuned XGB"]
print("Test performance comparison")
display(models_test_comp_df)


<p>Among all four models, the tuned Random Forest model was selected as the final model.</p>
<p>It achieved the highest test R-squared value of 0.9264, along with the lowest RMSE of 290.12 and the lowest MAE of 108.57. Although the tuned XGBoost model produced a slightly lower MAPE, the tuned Random Forest delivered the strongest overall test performance while also showing a much smaller train-test gap than the untuned models. This indicates that it generalizes better and is the most suitable model for deployment.</p>


<h2>Serialize the best model</h2>


In [ ]:
final_model = rf_best


In [ ]:
import os
import joblib

backend_sources = "backend_files"
saved_model = "forecast_superkart_sales_model.joblib"

if not os.path.exists(backend_sources):
    os.makedirs(backend_sources)

joblib.dump(final_model, os.path.join(backend_sources, saved_model))
print(f"Model saved at {os.path.join(backend_sources, saved_model)}")


In [ ]:
loaded_model = joblib.load(os.path.join(backend_sources, saved_model))
loaded_model


In [ ]:
loaded_model = joblib.load(os.path.join(backend_sources, saved_model))
loaded_model


<h1><strong>Deployment - Backend</strong></h1>


<h2>Flask Web Framework</h2>


<p>The backend will expose the trained machine learning model through a Flask API.</p>
<p>This API will:
- load the serialized final model
- accept product and store details as JSON input
- generate a sales prediction
- return the predicted sales value as a JSON response</p>
<p>Since the model was trained on the processed dataset, the API will expect the same final features used during model training.</p>


In [ ]:
backend_sources = "backend_files"

if not os.path.exists(backend_sources):
    os.makedirs(backend_sources)
    print(f"Directory '{backend_sources}' created successfully.")
else:
    print(f"Directory '{backend_sources}' already exists.")


<h2>Create the Flask API Application</h2>


In [ ]:
%%writefile "{backend_sources}/app.py"

import pandas as pd
import joblib
from flask import Flask, request, jsonify

# Initialize Flask app
app = Flask(__name__)

# Load the serialized final model
model = joblib.load("forecast_superkart_sales_model.joblib")

# Expected input columns based on the final processed training dataset
required_columns = [
    "Product_Weight",
    "Product_Sugar_Content",
    "Product_Allocated_Area",
    "Product_MRP",
    "Store_Size",
    "Store_Location_City_Type",
    "Store_Type",
    "Store_Age_Years",
    "Product_Type_Category",
    "Product_Id_char",
]

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "SuperKart Sales Forecast API is running",
        "prediction_endpoint": "/predict"
    })

@app.route("/predict", methods=["POST"])
def predict():
    try:
        payload = request.get_json(silent=True)

        if payload is None:
            return jsonify({"error": "Provide input data in JSON format."}), 400

        records = payload if isinstance(payload, list) else [payload]
        input_df = pd.DataFrame(records)

        missing_columns = [col for col in required_columns if col not in input_df.columns]
        if missing_columns:
            return jsonify({"error": f"Missing columns: {missing_columns}"}), 400

        input_df = input_df[required_columns]

        prediction = model.predict(input_df)

        return jsonify({
            "predictions": [round(float(value), 2) for value in prediction]
        })

    except Exception as e:
        return jsonify({"error": f"Prediction failed: {str(e)}"}), 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=7860)


<h2>Dependencies File</h2>


In [ ]:
%%writefile "{backend_sources}/requirements.txt"

pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
joblib==1.4.2
flask==3.0.3
gunicorn==23.0.0


<h2>Dockerfile</h2>


In [ ]:
%%writefile "{backend_sources}/Dockerfile"

FROM python:3.11-slim

WORKDIR /app

COPY . .

RUN pip install --no-cache-dir -r requirements.txt

CMD ["gunicorn", "--bind", "0.0.0.0:7860", "app:app"]


<h2>Setting up a Hugging Face Docker Space for the Backend</h2>
<p>To deploy the backend on Hugging Face:</p>
<ol>
<li>Go to Hugging Face and create a new Space</li>
<li>Select <strong>Docker</strong> as the SDK</li>
<li>Give the Space an appropriate name, such as <code>superkart-sales-backend</code></li>
<li>Keep the Space public or private based on your requirement</li>
<li>After creating the Space, note the repository ID in the format:</li>
</ol>
<p><code>your-username/your-space-name</code></p>
<p>This repository ID will be used in the upload step below.</p>


In [ ]:
backend_repo_id = "hazyn/SuperKart-Backend"
print("Update backend_repo_id before uploading files.")


<h2>Uploading Files to Hugging Face Space (Docker Space)</h2>


In [ ]:
# Save the final selected model into backend_files if not already saved there
joblib.dump(final_model, os.path.join(backend_sources, "forecast_superkart_sales_model.joblib"))

print("Final model saved inside backend_files.")
print(os.listdir(backend_sources))


In [ ]:
from huggingface_hub import login
import os

# Option 2:  using Colab secrets
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face successfully.")
else:
    print("HF_TOKEN is not set.")


In [ ]:
api = HfApi()

api.upload_folder(
    folder_path=backend_sources,
    repo_id=backend_repo_id,
    repo_type="space",
)

print("Backend files uploaded successfully.")


In [ ]:
import requests

base_url = "https://hazyn-superkart-backend.hf.space"
predict_url = f"{base_url}/predict"

print("Root check:")
root_response = requests.get(base_url)
print(root_response.status_code)
print(root_response.text)

sample_payload = {
    "Product_Weight": 12.66,
    "Product_Sugar_Content": "Low Sugar",
    "Product_Allocated_Area": 0.027,
    "Product_MRP": 117.08,
    "Store_Size": "Medium",
    "Store_Location_City_Type": "Tier 2",
    "Store_Type": "Supermarket Type2",
    "Store_Age_Years": 17,
    "Product_Type_Category": "Non Perishables",
    "Product_Id_char": "FD"
}

print("\nPredict check:")
pred_response = requests.post(predict_url, json=sample_payload)
print(pred_response.status_code)
print(pred_response.text)


<p>The backend API was successfully deployed on Hugging Face as a Docker Space.</p>
<p>The root endpoint confirmed that the Flask service was active, and the <code>/predict</code> endpoint successfully returned sales predictions for sample input data. This verifies that the trained model was correctly serialized, loaded in the backend application, and made accessible through a REST API.</p>


<h1><strong>Deployment - Frontend</strong></h1>


<h2>Points to note before executing the below cells</h2>
<ul>
<li>Create a Streamlit space on Hugging Face by following the instructions provided on the content page titled <strong><code>Creating Spaces and Adding Secrets in Hugging Face</code></strong> from Week 1</li>
</ul>


<h2>Streamlit for Interactive UI</h2>
<p>The frontend application will provide an interactive user interface for generating sales predictions.</p>
<p>This Streamlit app will:
- collect the required product and store inputs from the user
- send those inputs to the deployed backend API
- receive the predicted sales value
- display the result in a clean and interactive format</p>


In [ ]:
frontend_sources = "frontend_files"

if not os.path.exists(frontend_sources):
    os.makedirs(frontend_sources)
    print(f"Directory '{frontend_sources}' created successfully.")
else:
    print(f"Directory '{frontend_sources}' already exists.")


In [ ]:
%%writefile frontend_files/app.py
import streamlit as st
import requests

# --- Page Config ---
st.set_page_config(page_title="SuperKart Analytics Pro", page_icon="📈", layout="centered")

# --- Custom Styling ---
st.markdown("""
    <style>
    .input-card { background: #f8fafc; padding: 1.5rem; border-radius: 10px; border: 1px solid #e2e8f0; }
    .stNumberInput, .stSelectbox { margin-bottom: 1rem; }
    .header-text { text-align: center; color: #1e293b; margin-bottom: 2rem; }
    </style>
""", unsafe_allow_html=True)

st.markdown("<h1 class='header-text'>📈 SuperKart Sales Dashboard</h1>", unsafe_allow_html=True)

# --- Input Sections ---
col1, col2 = st.columns(2)

with col1:
    st.markdown("### 📦 Product Details")
    with st.container(border=True):
        Product_Weight = st.number_input("Product Weight (kg)", 0.0, 50.0, 12.66, help="Weight of the item in kg")
        Product_MRP = st.number_input("Maximum Retail Price ($)", 0.0, 500.0, 117.08, help="List price of the product")
        Product_Sugar_Content = st.selectbox("Sugar Content", ["Low Sugar", "Regular", "No Sugar"])
        Product_Id_char = st.selectbox("Category Prefix", ["FD", "NC", "DR"], help="FD=Food, NC=Non-Consumable, DR=Drink")
        Product_Type_Category = st.selectbox("Product Type", ["Perishables", "Non Perishables"])
        Product_Allocated_Area = st.number_input("Shelf Area Ratio", 0.0, 1.0, 0.027, format="%.3f", help="Percentage of shelf space allocated")

with col2:
    st.markdown("### 🏪 Store Context")
    with st.container(border=True):
        Store_Type = st.selectbox("Store Model", ["Supermarket Type1", "Supermarket Type2", "Departmental Store", "Food Mart"])
        Store_Size = st.selectbox("Store Size", ["Small", "Medium", "High"])
        Store_Location_City_Type = st.selectbox("City Tier", ["Tier 1", "Tier 2", "Tier 3"])
        Store_Age_Years = st.slider("Store Age (Years)", 0, 40, 17, help="How many years the store has been operating")

# --- Prediction Action ---
if st.button("🚀 GENERATE FORECAST", type="primary", use_container_width=True):
    payload = {
        "Product_Weight": Product_Weight, "Product_Sugar_Content": Product_Sugar_Content,
        "Product_Allocated_Area": Product_Allocated_Area, "Product_MRP": Product_MRP,
        "Store_Size": Store_Size, "Store_Location_City_Type": Store_Location_City_Type,
        "Store_Type": Store_Type, "Store_Age_Years": Store_Age_Years,
        "Product_Type_Category": Product_Type_Category, "Product_Id_char": Product_Id_char
    }

    with st.spinner("Analyzing market patterns..."):
        try:
            response = requests.post("https://hazyn-superkart-backend.hf.space/predict", json=payload, timeout=20)
            if response.status_code == 200:
                val = response.json()["predictions"][0]

                # Result Display
                st.divider()
                st.markdown(f"""
                    <div style="text-align: center; padding: 2rem; background: #ecfdf5; border-radius: 15px;">
                        <h3 style="color: #065f46;">PROJECTED WEEKLY SALES</h3>
                        <h1 style="font-size: 4rem; color: #059669;">${val:,.2f}</h1>
                    </div>
                """, unsafe_allow_html=True)
                st.balloons()
            else:
                st.error(f"Backend Error: {response.status_code}")
        except Exception:
            st.error("Could not reach the backend service.")


<h2>Dependencies File</h2>


In [ ]:
%%writefile "{frontend_sources}/requirements.txt"

streamlit==1.37.1
requests==2.32.4


<h2>DockerFile</h2>


In [ ]:
%%writefile "{frontend_sources}/Dockerfile"

FROM python:3.11-slim

WORKDIR /app

COPY . .

RUN pip install --no-cache-dir -r requirements.txt

# Force rebuild
EXPOSE 7860
CMD ["streamlit", "run", "app.py", "--server.port=7860", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false", "--server.headless=true"]


In [ ]:
frontend_repo_id = "hazyn/SuperKart-FrontEnd"
print("Update frontend_repo_id before uploading files.")


<h2>Uploading Files to Hugging Face Space (Streamlit Space)</h2>


In [ ]:
api = HfApi()

api.upload_folder(
    folder_path=frontend_sources,
    repo_id=frontend_repo_id,
    repo_type="space",
)

print("Frontend files uploaded successfully.")


<h1><strong>Actionable Insights and Business Recommendations</strong></h1>


<ul>
<li>
<p>The tuned Random Forest model delivered the strongest balance between predictive accuracy and generalization, making it the most suitable model for forecasting product-level store sales in this business setting.</p>
</li>
<li>
<p>Product price and shelf allocation both appear to influence revenue, which suggests that SuperKart should review pricing strategy together with in-store product placement rather than treating them as separate decisions.</p>
</li>
<li>
<p>Tier 2 stores generated the highest overall revenue in the analysis, indicating that these outlets may offer the best near-term opportunity for targeted promotions, inventory prioritization, and category expansion.</p>
</li>
<li>
<p>Supermarket formats, especially larger and better-performing store types, contribute more strongly to revenue than smaller formats. This suggests that store type should be factored into sales planning, stocking strategy, and campaign design.</p>
</li>
<li>
<p>Low Sugar products generated the highest total revenue in the dataset, which indicates that health-oriented or lower-sugar product lines may deserve stronger promotional visibility and inventory support.</p>
</li>
<li>
<p>Product categories such as Fruits and Vegetables and Snack Foods showed strong revenue contribution across stores, so these categories should be monitored closely for replenishment planning and sales optimization.</p>
</li>
<li>
<p>The engineered features, including <code>Store_Age_Years</code>, <code>Product_Type_Category</code>, and <code>Product_Id_char</code>, improved the usefulness of the data for modeling. This suggests that business-informed feature design can add real forecasting value beyond raw transactional variables alone.</p>
</li>
<li>
<p>The deployed backend API and frontend application make the forecasting system usable beyond analysis. SuperKart can now use the model not only for one-time predictions, but also as a reusable decision-support tool for future pricing, assortment, and inventory planning.</p>
</li>
<li>
<p>As a next step, SuperKart should collect more recent sales data and retrain the model periodically so that predictions continue to reflect changing customer demand, store performance, and product trends.</p>
</li>
</ul>
